# Editing models: every list of MODEL_LIST.md on the dev samples of its task

Every model of MODEL_LIST.md on a small fixed set of dev samples of the final dataset (`cryandrrich/editjudge`):
1. **Samples**: 30 dev samples per task (`python -m src.editors samples`); a sample belongs to one task only.
   Tasks: remove, add_replace (lists 2, 3), enhance, restore, change_color, formatting (lists 4 to 7) and
   open_ended (list 1 only). The generalists of list 1 run on every task.
2. **Plan** (change_color, formatting; remove and add_replace for their mask models): Qwen2.5-VL-7B names what to
   recolour, or the format action (crop, rotate, resize, extend, ...).
3. **Masks** (recolor, remove, add, replace): Grounding DINO + SAM 2.1.
4. **Run** every model in its own process (and its own environment when it pins other packages), **two at a time,
   one per T4**; a model that needs both GPUs (FLUX.2 klein) runs alone. While models run, a
   background thread **downloads the next models and builds their environments**, so the GPUs do not wait for the
   network; each model's weights are deleted after its last run (the disk is checked before every download). At most
   one large model (15 GB or more: VAREdit, FLUX.2 klein) runs at a time, for the 30 GB of RAM.
5. **Score** with the scorer (`mlp-siglip`), **report** per task and the **summary** of every task and model.

`SESSION` (cell 3):
- "quick": every model added since 2026-11-24 on `PER_EDITOR` samples of each task it applies to (2), a test of
  the whole flow. No new model starts after `MAX_HOURS`.
- "full": the models of `FULL_MODELS` on all 30 samples, after the quick test was checked.

How to run:
1. Inputs: the dataset `cryandrrich/editjudge`; for the scores, the output of `scorer_train` (it holds
   `out/scorer/runs/mlp-siglip/head.pt`); every earlier output of this notebook (plans, masks and outputs are
   reused, so a session goes on where the last one stopped). Settings: GPU T4 x2, internet on. Push the code first.
2. Secret `HF_TOKEN` (Add-ons > Secrets): optional (no model is gated since FLUX 3 Image Klein was dropped);
   it raises the Hugging Face download rate limit.
3. **Save Version > Save & Run All (Commit)**.

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_input_folders(tail, max_depth=6):
    """Folders under /kaggle/input whose path ends with `tail` (never inside a dataset's images/ folder)."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.endswith("/" + tail):
            found.append(root)
            dirs[:] = []
        elif root.count("/") - 2 >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [name for name in dirs if name != "images"]
    return found

DATA = find_dataset()
os.environ["EJ_DATA_DIR"] = DATA            # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT           # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"
os.makedirs(f"{OUT}/editors", exist_ok=True)
os.chdir(CODE)

# an earlier output of this notebook (to go on where it stopped), and the scorer's trained heads
for folder in find_input_folders("out/editors"):
    shutil.copytree(folder, f"{OUT}/editors", dirs_exist_ok=True)
    print("earlier editors output copied from", folder)
runs = [f for f in find_input_folders("scorer/runs") if glob.glob(f"{f}/mlp-siglip/head.pt")]
if runs:
    shutil.copytree(runs[0], f"{OUT}/scorer/runs", dirs_exist_ok=True)
    print("scorer heads copied from", runs[0])
else:
    print("no scorer_train output among the inputs: the scoring step is skipped")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install -U "transformers>=4.56,<5" "diffusers>=0.35,<0.40" "peft>=0.14" accelerate sentencepiece opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv
import threading, time
sys.path.insert(0, CODE)
STARTED = time.monotonic()
LOGS = f"{OUT}/editors/logs"
os.makedirs(LOGS, exist_ok=True)
print_lock = threading.Lock()

def say(text):
    with print_lock:
        print(f"[{(time.monotonic() - STARTED) / 3600:4.2f} h] {text}", flush=True)

def run_editors(*args, check=True, gpus=None, log=None):
    """python -m src.editors <args>. Without `log`, every line is printed as it comes; with it, the lines go to
    the log file and only a summary is printed. `gpus`: CUDA_VISIBLE_DEVICES for the process."""
    env = dict(os.environ, **({"CUDA_VISIBLE_DEVICES": gpus} if gpus is not None else {}))
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (out of memory on 2026-11-24)
    env.setdefault("USE_TF", "0")             # transformers would import Kaggle's TensorFlow (and its CUDA warnings)
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.editors", *map(str, args)], env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    handle = open(log, "a") if log else None
    for line in process.stdout:
        if handle:
            handle.write(line)
        else:
            print(line, end="", flush=True)
    if handle:
        handle.close()
    if process.wait() and check:
        raise subprocess.CalledProcessError(process.returncode, args)
    return process.returncode

def free_gb():
    return shutil.disk_usage("/tmp").free / 1e9

def drop_cache(*repos):
    """Delete downloaded Hugging Face repositories that are no longer needed."""
    for repo in repos:
        shutil.rmtree(f"/tmp/hf/hub/models--{repo.replace('/', '--')}", ignore_errors=True)

INFO = {row["name"]: row for row in json.loads(subprocess.run(
    [sys.executable, "-m", "src.editors", "info"], capture_output=True, text=True, check=True).stdout)}
say(f"{len(INFO)} editors; {free_gb():.0f} GB free in /tmp")

In [ ]:
# What this session runs:
# - "preflight": NO GPU (Accelerator: None; free): builds every environment, downloads every model (then deletes it)
#   and runs the small models on 1 sample per task on the CPU, so broken pins, wrong file names or a missing licence
#   show up before any GPU time is spent;
# - "quick": every new model on a few samples of each task (2 x T4, about 6 h at most);
# - "full": the models of FULL_MODELS on all 30 samples, after the quick test was checked.
SESSION = "quick"
CPU_MODELS = ["realesrgan", "hvi-cidnet", "instructir", "codeformer", "gfpgan", "ddcolor", "ben2", "birefnet"]
PER_EDITOR = 2                            # quick: samples per task each model applies to
PER_EDITOR_OF = {}                        # model -> its own PER_EDITOR
MAX_HOURS = 5.5                           # no new model starts after this (the running ones finish)
DISK_MARGIN_GB = 12                       # free space kept in /tmp besides the next download
# Order (also the download order): the small models first, then the medium ones; the large ones after (one at a
# time); the two-GPU model last. InstantIR, OmniGen2, VIBE and FLUX 3 Klein were dropped (too slow for live use)
QUICK_MODELS = {                          # model -> its tasks in the quick test (lists 2 and 3 ran fully on 2026-11-21)
    "realesrgan": ["enhance", "formatting"], "hvi-cidnet": ["enhance"], "instructir": ["enhance"],
    "codeformer": ["restore"], "gfpgan": ["restore"], "ddcolor": ["change_color"], "ben2": ["formatting"],
    "birefnet": ["formatting"], "plain-code": ["formatting"],
    "instruct-pix2pix": "all", "magicbrush": "all", "realedit": "all", "vosr": ["enhance", "restore"],
    "sdxl-inpaint": ["change_color"], "sdxl-outpaint": ["formatting"], "varedit-2b": "all",
    "flux2-klein-4b": "all",
}
FULL_MODELS = {}                          # "full": model -> tasks ("all" = every task), after the quick test
# Only these models of the session (empty: all of them). Quick run 2 of 2026-11-24: the two that failed in run 1
# and are still candidates; attach run 1's output as an input so the finished models are scored with them
ONLY = ["plain-code", "varedit-2b"]
ALL_TASKS = ["remove", "add_replace", "enhance", "restore", "change_color", "formatting", "open_ended"]
PLAN_TASKS = ["change_color", "formatting"]   # plans needed by the models above (remove, add_replace: lists 2, 3)
MASK_TASKS = ["change_color"]

chosen = FULL_MODELS if SESSION == "full" else QUICK_MODELS
JOBS = [(model, ALL_TASKS if tasks == "all" else tasks) for model, tasks in chosen.items() if not ONLY or model in ONLY]
for model, tasks in JOBS:
    assert model in INFO and set(tasks) <= set(INFO[model]["tasks"]), model
say("jobs: " + ", ".join(f"{model} ({len(tasks)})" for model, tasks in JOBS))

In [ ]:
# The scheduler (used in cell 5). A prefetch thread downloads weights and builds environments ahead of the runs; it
# starts here, so the first models download while Qwen makes the plans; two GPU slots then run the models
from src.common.envs import ensure_env

ready = {model: threading.Event() for model, _ in JOBS}
fetch_failed, fetched, done, results = {}, set(), set(), []
MAX_AHEAD = 3                             # models downloaded but not finished (2 running, 1 waiting)
slots = {"0": None, "1": None}            # GPU -> the model running there
state_lock = threading.Condition()

def is_large(model):
    return INFO[model]["download_gb"] >= 15

def deadline_passed():
    return (time.monotonic() - STARTED) / 3600 > MAX_HOURS

def prefetch():
    """Downloads and environments in job order, at most MAX_AHEAD models on disk, and only when the disk has room
    for the download (or when nothing else is on disk: then it has to try)."""
    for model, _ in JOBS:
        while True:
            with state_lock:
                holding = len(fetched - done)
            need = INFO[model]["download_gb"] + DISK_MARGIN_GB
            if (holding < MAX_AHEAD and free_gb() >= need) or holding == 0 or deadline_passed():
                break
            time.sleep(5)
        if deadline_passed():
            break
        started = time.monotonic()
        try:
            with open(f"{LOGS}/env-{model}.log", "w") as log:
                ensure_env(tuple(INFO[model]["install"]), log, INFO[model]["isolated"])
            if run_editors("fetch", "--model", model, check=False, log=f"{LOGS}/{model}.log"):
                fetch_failed[model] = "fetch failed"
        except Exception as error:
            fetch_failed[model] = f"environment: {error}"
        with state_lock:
            fetched.add(model)
        say(f"prefetched {model} in {(time.monotonic() - started) / 60:.1f} min"
            + (f" ({fetch_failed[model]})" if model in fetch_failed else "") + f"; {free_gb():.0f} GB free")
        ready[model].set()

def clean_up(model):
    """Delete the model's downloads unless a job that has not finished needs them too."""
    with state_lock:
        later = [other for other, _ in JOBS if other not in done and other != model]
    keep_repos = {repo for other in later for repo in INFO[other]["repos"]}
    keep_dirs = {folder for other in later for folder in INFO[other]["local_dirs"]}
    drop_cache(*[repo for repo in INFO[model]["repos"] if repo not in keep_repos])
    for folder in INFO[model]["local_dirs"]:
        if folder not in keep_dirs:
            shutil.rmtree(f"/tmp/models/{folder}", ignore_errors=True)

def run_job(model, tasks, gpus):
    ready[model].wait()
    started = time.monotonic()
    if model in fetch_failed:
        results.append((model, "-", fetch_failed[model], 0.0))
    else:
        limit = PER_EDITOR_OF.get(model, PER_EDITOR) if SESSION == "quick" else None
        # all its tasks in one process: the model loads once (counts per task: the summary of cell 6)
        code = run_editors("runall", "--tasks", ",".join(tasks), "--models", model,
                           *(["--per-editor", limit] if limit else []), check=False, gpus=gpus,
                           log=f"{LOGS}/{model}.log")
        results.append((model, ",".join(tasks), "ok" if code == 0 else "FAILED", (time.monotonic() - started) / 60))
        say(f"{model}: {'ok' if code == 0 else 'FAILED'} (gpu {gpus})")
    with state_lock:
        done.add(model)
    clean_up(model)
    say(f"finished {model} in {(time.monotonic() - started) / 60:.1f} min; {free_gb():.0f} GB free")

def worker(model, tasks, gpus):
    try:
        run_job(model, tasks, gpus)
    finally:
        with state_lock:
            for gpu in gpus.split(","):
                slots[gpu] = None
            state_lock.notify_all()

def preflight():
    """One model after another: environment, download, a CPU run of the small ones (1 sample per task), clean up."""
    for model, tasks in JOBS:
        started = time.monotonic()
        try:
            with open(f"{LOGS}/env-{model}.log", "w") as log:
                ensure_env(tuple(INFO[model]["install"]), log, INFO[model]["isolated"])
            status = "env ok"
        except Exception as error:
            results.append((model, "-", f"environment failed: {error}"[:80], 0.0))
            say(f"{model}: environment FAILED (log env-{model}.log)")
            continue
        if run_editors("fetch", "--model", model, check=False, log=f"{LOGS}/{model}.log"):
            status = "fetch FAILED"
        elif model in CPU_MODELS:
            for task in tasks:
                if INFO[model]["uses_plan"] and task in PLAN_TASKS:
                    continue                          # no plans in preflight (they need Qwen on a GPU)
                code = run_editors("runall", "--task", task, "--models", model, "--per-editor", 1, "--device",
                                   "cpu", check=False, log=f"{LOGS}/{model}.log")
                status = "cpu run ok" if code == 0 and status != "cpu run FAILED" else "cpu run FAILED"
        else:
            status = "env and download ok"
        minutes = (time.monotonic() - started) / 60
        results.append((model, ",".join(tasks), status, minutes))
        with state_lock:
            done.add(model)
        clean_up(model)
        say(f"{model}: {status} ({minutes:.1f} min); {free_gb():.0f} GB free")


def pick(pending):
    """The first pending job that is downloaded and fits the free GPUs and the large-model rule, with its GPUs."""
    free = [gpu for gpu, value in slots.items() if value is None]
    large_busy = any(value is not None and is_large(value) for value in slots.values())
    for model, tasks in pending:
        if not ready[model].is_set() or (is_large(model) and large_busy):
            continue
        if INFO[model]["gpus"] == 2 and len(free) == 2:
            return model, tasks, "0,1"
        if INFO[model]["gpus"] == 1 and free:
            return model, tasks, free[0]
    return None

def run_jobs():
    """Two GPU slots, models started as their downloads finish (the prefetch thread started in this cell)."""
    threads, pending = [], list(JOBS)
    while pending and not deadline_passed():
        with state_lock:
            choice = pick(pending)
            if choice is None:
                state_lock.wait(timeout=5)    # a slot frees up or a download finishes
                continue
            model, tasks, gpus = choice
            for gpu in gpus.split(","):
                slots[gpu] = model
        pending.remove((model, tasks))
        say(f"start {model} on gpu {gpus}")
        thread = threading.Thread(target=worker, args=(model, tasks, gpus))
        thread.start()
        threads.append(thread)
    if pending:
        say("time limit: not started: " + ", ".join(model for model, _ in pending))
    for thread in threads:
        thread.join()


if SESSION != "preflight":
    threading.Thread(target=prefetch, daemon=True).start()

# 1-3. samples (fixed by the dataset; in this order, so a sample stays in one task), plans (Qwen over both GPUs)
# and masks; finished ones are skipped. Meanwhile the prefetch thread downloads the first models.
for task in ALL_TASKS:
    run_editors("samples", "--task", task)
if SESSION == "preflight":
    PLAN_TASKS, MASK_TASKS = [], []          # Qwen, DINO and SAM need the GPU
for task in PLAN_TASKS:
    run_editors("plan", "--task", task)
drop_cache("Qwen/Qwen2.5-VL-7B-Instruct")
for task in MASK_TASKS:
    run_editors("masks", "--task", task)
drop_cache("IDEA-Research/grounding-dino-base", "facebook/sam2.1-hiera-large")
say(f"plans and masks done; {free_gb():.0f} GB free")

In [ ]:
# 4. the models: two at a time (one per T4), each as soon as its download is ready; a table of every run at the end
if SESSION == "preflight":
    preflight()
else:
    run_jobs()
    say("all runs finished")
for model, task, status, minutes in results:
    print(f"{model:17} {task:13} {status:8} {minutes:6.1f} min")
print("not ok:", sorted({model for model, _, status, _ in results if "ok" not in status}) or "none",
      "- logs in out/editors/logs/")

In [ ]:
# 5. scores, reports and the summary of every task (the scorer needs the scorer_train output among the inputs)
from IPython.display import Image, Markdown, display
for task in ALL_TASKS if SESSION != "preflight" else []:
    if glob.glob(f"{OUT}/scorer/runs/mlp-siglip/head.pt"):
        run_editors("score", "--task", task, check=False)
    run_editors("report", "--task", task, check=False)
run_editors("summary")
display(Markdown(open(f"{OUT}/editors/summary.md").read()))
say("done")

In [ ]:
# Last cell: the clone of the code is removed (the saved output keeps only results), and the whole out/ folder
# goes into one zip next to it: download /kaggle/working/out.zip. The output shows real people: team only.
os.chdir("/kaggle/working")
shutil.rmtree(CODE, ignore_errors=True)
archive = shutil.make_archive("/kaggle/working/out", "zip", "/kaggle/working", "out")
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")